## Shared Config (if needed)

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MultipleLocator

from ravelry_scraper.constants import DATA_DIR


In [ ]:
con = duckdb.connect()

parquet_files = list(DATA_DIR.glob("**/*.parquet"))

# Register Parquet datasets as SQL views
for file in parquet_files:
    name = f"{file.parent.name}_{file.stem}"
    con.execute(f"""
        CREATE OR REPLACE VIEW {name} AS
        SELECT *
        FROM read_parquet('{file}')
    """)


In [ ]:
# Store list of all DuckDB views
duckdb_views = [row[0] for row in con.execute("""
    SELECT view_name
    FROM duckdb_views
""").fetchall()]

# Print all DuckDB views to verify the list
print(duckdb_views)

In [ ]:
pl.Config.set_fmt_float("full")
pl.Config.set_tbl_rows(25)
pl.Config.set_thousands_separator(",")

# Designers (aka PatternAuthors)

Ravelry's API exposes an endpoint for `GET /designers/{id}.json` that returns Designer (aka PatternAuthor) records. A Designer can be associated with more than 1 User but typically is only associated to 1. Cleaning in Python focused on dropping unused columns, standardizing data types, and extracting nested records (e.g., Users) into separate tables, then storing just the IDs in their place. From the cleaned data, notable columns include:

| **Notable Fields** | **Use Cases** |
| --- | --- |
| `id`, `name`, `user_ids` | Identify Designers and join them to Users |
| `patterns_count`, `crochet_pattern_count`, `knitting_pattern_count` | Evaluate Designer productivity and craft preference |
| `favorites_count` | Analyze Designer popularity |

### Documentation
- https://www.ravelry.com/api#designers_show
- https://www.ravelry.com/api#PatternAuthor_full_result

## Data Cleaning

First, we can confirm that cleaning the data in Python did not remove any records:

In [ ]:
display(
    con.sql("""
        SELECT
            (SELECT COUNT(*) FROM raw_designers) AS raw_count,
            (SELECT COUNT(*) FROM clean_designers) AS clean_count,
            (SELECT (SELECT COUNT(*) FROM raw_designers) - (SELECT COUNT(*) FROM clean_designers)) AS difference
    """).pl()
)

Second, we can compare the two schemas to see which columns were dropped and how data types were standardized and / or extracted into separate tables:

In [ ]:
display(
    con.sql("""
        WITH raw_schema AS (
            DESCRIBE raw_designers
        ),
        clean_schema AS (
            DESCRIBE clean_designers
        )
        SELECT 
            COALESCE(raw_schema.column_name, clean_schema.column_name) AS column_name,
            CASE
                WHEN raw_schema.column_type LIKE 'STRUCT%' THEN 'STRUCT()'
                ELSE COALESCE(raw_schema.column_type, '<EXTRACTED>')
            END AS raw_type,
            CASE
                WHEN raw_schema.column_type LIKE 'STRUCT%' THEN '<EXTRACTED>'
                ELSE COALESCE(clean_schema.column_type, '<DROPPED>')
            END AS clean_type,
        FROM raw_schema
        FULL OUTER JOIN clean_schema
        ON raw_schema.column_name = clean_schema.column_name
    """).df()
)


By viewing a few lines, we can get a better sense for the dataset we will be working with:

In [ ]:
display(
    con.sql("""
        SELECT * 
        FROM clean_designers
        LIMIT 10
    """).pl()
)

## Data Exploration

To begin the data exploration, we look at basic summary statistics. Note that this data was scraped on November 15, 2026. Numbers may be different if you have run the scraper more recently.

Our dataset has 122,132 Designers. Of those, 112,891 have published at least one Pattern, for a total of 1,537,966 Patterns. This differs slightly from the Patterns dataset, which was scraped more recently.

In [ ]:
pdf1 = con.sql("""
    SELECT
        COUNT(*) AS num_designers,
        COUNT(CASE WHEN patterns_count > 0 THEN 1 END) AS with_patterns,
        SUM(patterns_count) AS sum_designer_patterns,
        MAX(scraped_at) AS designers_last_scraped_at
    FROM clean_designers
""").pl()

pdf2 = con.sql("""
    SELECT
        COUNT(*) AS count_patterns_dataset,
        MAX(scraped_at) AS patterns_last_scraped_at
    FROM clean_patterns
""").pl()

pdf = pl.concat([pdf1, pdf2], how="horizontal")
pdf = pdf.with_columns(
    (pl.col("count_patterns_dataset") - pl.col("sum_designer_patterns")).alias("patterns_diff"),
    (pl.col("patterns_last_scraped_at") - pl.col("designers_last_scraped_at")).alias("patterns_diff_time")
)
pdf.show()

For summary statistics covering Pattern information, we want to separate the Designers without any Patterns:

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW designers AS (
        SELECT *
        FROM clean_designers
        WHERE patterns_count > 0
    )
""")

### Summary Statistics

The Designer dataset contains 112,891 designers with at least 1 Pattern, representing 92.4% of the original 122,132 Designer records. Their individual Pattern catalog counts sum to 1,537,966 Patterns.

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW designer_stats AS
        SELECT
            COUNT(*) AS num_designers,
            SUM(patterns_count) AS num_patterns,
            ROUND(
                AVG(patterns_count)
            , 2) AS avg_num_patterns,
            MEDIAN(patterns_count) AS median_num_patterns,
            MAX(patterns_count) AS max_num_patterns,
            COUNT(knitting_pattern_count) AS num_knitting_patterns,
            ROUND(
                AVG(knitting_pattern_count)
            , 2) AS avg_num_knitting_patterns,
            MEDIAN(knitting_pattern_count) AS median_num_knitting_patterns,
            MAX(knitting_pattern_count) AS max_num_knitting_patterns,
            COUNT(crochet_pattern_count) AS num_crochet_patterns,
            ROUND(
                AVG(crochet_pattern_count)
            , 2) AS avg_num_crochet_patterns,
            MEDIAN(crochet_pattern_count) AS median_num_crochet_patterns,
            MAX(crochet_pattern_count) AS max_num_crochet_patterns,
            MAX(LENGTH(user_ids)) AS max_user_ids_length,
            MODE(LENGTH(user_ids)) AS mode_user_ids_length,
            SUM(favorites_count) AS num_favorites,
            ROUND(
                AVG(favorites_count)
            , 2) AS avg_num_favorites,
            MEDIAN(favorites_count) AS median_num_favorites,
            MAX(favorites_count) AS max_num_favorites
        FROM designers
        WHERE patterns_count > 0
""")

designer_summary_df = con.sql("""
    SELECT *
    FROM designer_stats
""").pl()

display(designer_summary_df.transpose(include_header=True))

The median catalog contains 2 Patterns, while the maximum catalog contains 13,517 Patterns. The mean is 13.62 Patterns. This suggests a heavily right-skewed distribution of catalog sizes, with most Designers having only a couple Patterns but a few having exceptionally large catalogs.

The median Designer has 1 knitting Pattern and 1 crochet Pattern, but knitting catalogs are larger on average with 7.88 knitting Patterns compared to 5.75 crochet Patterns. The largest knitting catalog contains 11,345 knitting Patterns, and the largest crochet catalog contains 2,988 crochet Patterns. While knitting and crochet catalogs have the same median size, knitting catalogs are larger on average and reach a much higher maximum.

Additionally, Ravelry allows users to "favorite" their favorite Designers. Across the Designers with at least 1 Pattern, Ravelry reports 7,959,235 total favorites. The median is 1 favorite, so most of the Designers have only 1 favorite or fewer. The average is 70.5 favorites per Designer, with a maximum of 71,248 favorites. This also displays a right-skewed distribution, with most Designers receiving only a few favorites but the most popular Designers receiving tens of thousands of favorites.

### Most Popular Designers

In [ ]:
df = con.sql("""
    SELECT
        name,
        favorites_count
    FROM designers
    ORDER BY favorites_count DESC
    LIMIT 10
""").pl()

display(df)

total_favorites = con.sql("""
    SELECT num_favorites
    FROM designer_stats
""").pl().item()

print("Percentage of total favorites:", round(df.select("favorites_count").sum().item() / total_favorites * 100, 2), "%")

Ranked using favorite counts, PetiteKnit is the most popular Designer with 71,248 favorites. This is about 30% more than the 2nd most popular Designer, Andrea Mowry. The 10th-ranked Designer, Purl Soho, has 28,706 favorites.

All together, the top 10 most-favorited Designers account for 5.51% of favorites.

### Most Prolific Designers

In [ ]:
df = con.sql("""
    SELECT
        name,
        patterns_count
    FROM designers
    ORDER BY patterns_count DESC
    LIMIT 10
""").pl()

display(df)

total_patterns = con.sql("""
    SELECT num_patterns
    FROM designer_stats
""").pl().item()

print("Percentage of total patterns:", round(df.select("patterns_count").sum().item() / total_patterns * 100, 2), "%")

Examining the catalogs, DROPS design has the largest catalog with 13,517 Patterns. This is more than double the number of Patterns in the 2nd-largest catalog, produced by Bergère de France.

Several entries represent major brands or design teams, like Lion Brand Yarn.

All together, the top 10 most prolific Designers' catalogs represent 3.58% of all Patterns.

### Most Popular Designer's Most Popular Patterns

Looking now at the most popular designer, PetiteKnit, we will examine her most popular Patterns.

Like with Designers, Ravelry allows Users to "favorite" Patterns. First, we will evaluate PetiteKnit's catalog via favorites. Second, we will evaluate her catalog via Projects.

#### By Favorites

In [ ]:
pop_by_favs_df = con.sql("""
    WITH most_popular_designer AS (
        SELECT
            id,
            name,
            favorites_count
        FROM designers
        ORDER BY favorites_count DESC
        LIMIT 1
    )
    SELECT
        designer.name,
        pattern.name AS pattern_name,
        pattern.favorites_count AS num_favorites
    FROM most_popular_designer AS designer
    JOIN clean_patterns AS pattern
        ON designer.id = pattern.designer_id
    ORDER BY pattern.favorites_count DESC
    LIMIT 10
""").pl()

display(pop_by_favs_df)

#### By Projects

In [ ]:
pop_by_projects_df = con.sql("""
    WITH most_popular_designer AS (
        SELECT
            id,
            name,
            favorites_count
        FROM designers
        ORDER BY favorites_count DESC
        LIMIT 1
    )
    SELECT
        designer.name,
        pattern.name AS pattern_name,
        pattern.projects_count AS num_projects
    FROM most_popular_designer AS designer
    JOIN clean_patterns AS pattern
        ON designer.id = pattern.designer_id
    ORDER BY pattern.projects_count DESC
    LIMIT 10
""").pl()

display(pop_by_projects_df)

#### Comparison

In [ ]:
df = pl.concat([
    pop_by_favs_df.drop("name").select(pl.all().name.prefix("fav_")),
    pop_by_projects_df.drop("name").select(pl.all().name.prefix("proj_"))
], how="horizontal")

display(df)

#### Discussion

The No Frills Sweater ranks 1st by favorites and 5th by Projects. The Sophie Scarf ranks 1st by Projects and 2nd by favorites. In fact, the Sophie Series (Scarf, Hood, and Shawl) occupy 3 of the top 4 spots, though the Sophie Shawl does not appear in the favorites top 10 list. Notably, 7 Patterns appear across both top 10 lists.

From personal domain knowledge, the Sophie Series are some of the most widely-known and widely-knit Patterns. Therefore, while favorites is a good measure of interest and inspiration, Projects is a better measure of real-world output and popularity.

#### Correlation Matrix


In [ ]:
metrics_df = con.sql("""
    WITH designer_age AS (
        SELECT
            designer_id,
            MIN(EXTRACT(YEAR FROM created_at)) AS first_year
        FROM clean_patterns
        GROUP BY designer_id
    )
    SELECT
        first_year,
        patterns_count,
        favorites_count
    FROM designers
    LEFT JOIN designer_age
        ON designers.id = designer_age.designer_id
""").df()

corr_matrix = metrics_df.corr(method="spearman")

fig, axes = plt.subplots(figsize=(15, 6))
heatmap_settings = {
    "annot": True,
    "fmt": ".2f",
    "cmap": sns.blend_palette([
        "#e7f0f7",
        "#5a0e21"
    ], as_cmap=True),
    "square": True,
    "vmax": 1,
    "vmin": -1,
    "center": 0,
}

sns.heatmap(corr_matrix, **heatmap_settings)
plt.title("Designer Output and Popularity")

plt.tight_layout()
plt.show()


The correlation matrix shows that Designers with larger catalogs tend to have more favorites: Catalog size (patterns_count) and Designer favorites (favorites_count) have a strong positive correlation of 0.71.

A Designer's "age" (year of first Pattern creation) has a weak relationship with both catalog size (-0.13) and favorites (-0.10). As might be expected, older Designers tend to have larger catalogs and more favorites, but only slightly.

### Most Popular Designer Patterns Per Year

Looking at our top 10 most popular Designers (via favorites), what does their Pattern output look like each year?

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW popular_per_year AS
    WITH pattern_years AS (
        SELECT
            id,
            designer_id,
            YEAR(created_at) AS year
        FROM clean_patterns
    ),
    most_popular_designers AS (
        SELECT
            id,
            name
        FROM clean_designers
        ORDER BY favorites_count DESC
        LIMIT 10
    )
    SELECT
        designer.name AS designer_name,
        pattern.id AS pattern_id,
        pattern.year AS pattern_year,
    FROM most_popular_designers AS designer
    JOIN pattern_years AS pattern
        ON designer.id = pattern.designer_id
""")

display(
    con.sql("""
        PIVOT popular_per_year
        ON pattern_year
        USING COALESCE(COUNT(pattern_id), 0)
        GROUP BY designer_name
    """).pl()
)

In [ ]:
popular_per_year_df = con.sql("""
    SELECT
        designer_name,
        pattern_year,
        COUNT(pattern_id) AS pattern_count,
        SUM(COUNT(pattern_id)) OVER (
            PARTITION BY designer_name 
            ORDER BY pattern_year
        ) AS running_total
    FROM popular_per_year
    GROUP BY designer_name, pattern_year
    ORDER BY designer_name, pattern_year
""").df()

plt.figure(figsize=(12, 6))
ax = sns.lineplot(
    data=popular_per_year_df,
    x="pattern_year",
    y="running_total",
    hue="designer_name",
    marker="o"
)
ax.xaxis.set_major_locator(MultipleLocator(2))
ax.xaxis.set_major_formatter(
    FuncFormatter(lambda year, _: "2026\nYTD" if year == 2026 else f"{year:.0f}")
)

plt.title("Cumulative Patterns Published per Year by Favorite Designers")
plt.xlabel("Year")
plt.ylabel("Pattern Count")

plt.legend(title="Designer", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

Purl Soho has the most total Patterns across the top 10 most-favorited Designers, and they are also the only top 10 Designer with patterns published each year since Ravelry's founding.

Sari Nordlund has the highest single-year output of the top 10 Designers, with 78 Patterns produced in 2021. Stephen West came close in second with 76 Patterns produced in 2023 and 72 in 2025.

PetiteKnit, the top-favorited Designer, first appears in 2016 and peaks at 51 Patterns produced in 2021.

### Most Prolific Designer Patterns Per Year

Looking at our top 10 most prolific Designers, what does their Pattern output look like each year?

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW prolific_per_year AS
    WITH pattern_years AS (
        SELECT
            id,
            designer_id,
            YEAR(created_at) AS year
        FROM clean_patterns
    ),
    most_prolific_designers AS (
        SELECT
            id,
            name
        FROM clean_designers
        ORDER BY patterns_count DESC
        LIMIT 10
    )
    SELECT
        designer.name AS designer_name,
        pattern.id AS pattern_id,
        pattern.year AS pattern_year,
    FROM most_prolific_designers AS designer
    JOIN pattern_years AS pattern
        ON designer.id = pattern.designer_id
""")

display(
    con.sql("""
        PIVOT prolific_per_year
        ON pattern_year
        USING COALESCE(COUNT(pattern_id), 0)
        GROUP BY designer_name
    """).pl()
)

In [ ]:
prolific_per_year_df = con.sql("""
    SELECT
        designer_name,
        pattern_year,
        COUNT(pattern_id) AS pattern_count
    FROM prolific_per_year
    GROUP BY designer_name, pattern_year
    ORDER BY designer_name, pattern_year
""").df()

plt.figure(figsize=(12, 6))
ax = sns.lineplot(
    data=prolific_per_year_df,
    x="pattern_year",
    y="pattern_count",
    hue="designer_name",
    marker="o"
)
ax.xaxis.set_major_locator(MultipleLocator(2))
ax.xaxis.set_major_formatter(
    FuncFormatter(lambda year, _: "2026\nYTD" if year == 2026 else f"{year:.0f}")
)

plt.title("Patterns Published per Year by Most Prolific Designers")
plt.xlabel("Year")
plt.ylabel("Pattern Count")
# Move legend outside the plot so it doesn't cover the lines
plt.legend(title="Designer", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = prolific_per_year_df.sort_values("pattern_year", ascending=True)
plot_df["pattern_year"] = plot_df["pattern_year"].astype(int)

fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(
    data=plot_df,
    x="pattern_year",
    weights="pattern_count",
    hue="designer_name",
    multiple="stack",
    discrete=True,
    shrink=0.8,
    ax=ax
)
ax.xaxis.set_major_locator(MultipleLocator(1))
ax.xaxis.set_major_formatter(
    FuncFormatter(lambda year, _: "2026\nYTD" if year == 2026 else f"{year:.0f}")
)
ax.set_xlim(
    plot_df["pattern_year"].min() - 0.5,
    plot_df["pattern_year"].max() + 0.5
)
ax.set(xlabel="Year", ylabel="Patterns published")

sns.move_legend(
    ax,
    "upper left",
    bbox_to_anchor=(1.02, 1),
    title="Designer",
)

plt.tight_layout()
plt.show()

In [ ]:
cumulative_prolific_per_year_df = con.sql("""
    SELECT
        designer_name,
        pattern_year,
        COUNT(pattern_id) AS pattern_count,
        SUM(COUNT(pattern_id)) OVER (
            PARTITION BY designer_name 
            ORDER BY pattern_year
        ) AS running_total
    FROM prolific_per_year
    GROUP BY designer_name, pattern_year
    ORDER BY designer_name, pattern_year
""").df()

plt.figure(figsize=(12, 6))
ax = sns.lineplot(
    data=cumulative_prolific_per_year_df,
    x="pattern_year",
    y="running_total",
    hue="designer_name",
    marker="o"
)
ax.xaxis.set_major_locator(MultipleLocator(2))
ax.xaxis.set_major_formatter(
    FuncFormatter(lambda year, _: "2026\nYTD" if year == 2026 else f"{year:.0f}")
)

plt.title("Cumulative Patterns Published by Most Prolific Designers")
plt.xlabel("Year")
plt.ylabel("Total Pattern Count")
# Move legend outside the plot so it doesn't cover the lines
plt.legend(title="Designer", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

DROPS design, the most prolific Designer, maintained consistently high output throughout the entire timeframe, with more than 400 Patterns every year and a peak of 1,392 Patterns in 2007 (which likely reflected the initial back catalog upload).

Other Designers had more pronounced spikes. For example, Bergère de France produced 2,007 Patterns in 2021, which is the highest single-year output in the table. Lion Brand Yarn peaked at 1,235 Patterns in 2013 but has produced less than 100 Patterns per year since 2015.

Among the past few years, King Cole Ltd produced 993 Patterns in 2023, and Lana Grossa produced 1,322 Patterns in 2024. In 2025, Lana Grossa produced 799 Patterns, King Cole Ltd produced 794 Patterns, and OZ Verlag Design Team produced 780 Patterns - a difference of only 19 Patterns. This year, OZ Verlag Design Team leads with 738 Patterns to date.

## Challenges and Limitations

No major challenges were encountered in the wrangling and cleaning of the Designer dataset.

However, the Designer dataset may be of limited usefulness. The only unique value it provides is the `favorites_count` column. Other potentially useful columns are summary statistics that may quickly become dated.


### Outdated Summary Columns

In [ ]:
con.sql("""
    WITH differences AS (
        SELECT
            designer.name,
            designer.patterns_count,
            COUNT(pattern.*) AS num_patterns_dataset,
            designer.patterns_count - COUNT(pattern.*) AS diff
        FROM clean_designers AS designer
        JOIN clean_patterns AS pattern
            ON designer.id = pattern.designer_id
        GROUP BY designer.name, designer.patterns_count
        HAVING diff <> 0
    )
    SELECT
        COUNT(*) AS num_designers,
        SUM(ABS(diff)) AS total_pattern_diff,
        ROUND(AVG(ABS(diff)), 2) AS avg_pattern_diff
    FROM differences
""").pl()

Columns like `favorites_count` and `patterns_count` may quickly become dated. These numbers can change daily, but the scraper currently does not support periodically refreshing previously-fetched data.

Comparing the Designer `patterns_count` column against the Patterns dataset itself, we find over 3,600 Patterns are unaccounted for in the Designers dataset. This occurred over a span of 2 weeks, meaning approximately 186 new Patterns are uploaded each day. It is uncertain how often the `favorites_count` may change, but, given there are over 9 million users, we believe this value changes daily as well.

Updating the pipelines to periodically refresh data is an area for future improvement. We could also store counts like `favorites_count` in a time series, to analyze should the number ever decreases.